# 00. SageMaker Studio setup and run management

UTS 43008 Reinforcement Learning · Assignment 3 · Project 12 · Dihedrals

Check the kernel, project files and storage before spending time on training.
The project uses local files in your persistent Studio project folder. It does not use S3.

The current default study has twenty full learners, preceded by a separate pilot pair and team
review. Read the [team master](../docs/TEAM_MASTER.md) and [runbook](../docs/RUNBOOK.md) before
starting that workflow. The study controller and evaluation journals support compatible resumes.

The supplied Studio screenshots show these choices:

| Instance | CPU / system RAM | GPU | Role |
|---|---|---|---|
| ml.t3.medium | 2 vCPU / 4 GiB | None | Light reading and checks |
| ml.g4dn.xlarge | 4 vCPU / 16 GiB | One T4 | Starting point for the pilot |
| ml.g4dn.2xlarge | 8 vCPU / 32 GiB | One T4 | More CPU/RAM if measured simulation load needs it |

The screenshots show 50 GB of persistent storage and an administrator-controlled 60-minute idle
setting. That is not ordinarily a one-hour training limit; confirm the course lifecycle script's
behaviour rather than disabling it. Read the image-upgrade warning and back up files before
updating the image. A GPU selection alone does not prove this kernel has compatible CUDA.

Read 00, 01 and 02 first. Complete the short recovery lab in 03b before starting full experiments
in 03. Notebooks 04 and 05 cover evaluation and the demonstration.

## Before the first code cell

Open the complete project, not just the notebooks. Run these commands in a Studio terminal from
the project folder. They install a project environment; this notebook does not run them automatically.

~~~bash
uv sync --frozen
uv run python -m ipykernel install --user --name base-siege --display-name "Python 3 (Base Siege)"
~~~

Select **Python 3 (Base Siege)** in the kernel menu. Installation needs internet access, Python
3.12 and enough disk space. A GPU instance alone does not prove that the kernel can use CUDA.

In [ ]:
from pathlib import Path
import os
import sys
import json
import uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

candidate = Path(os.environ.get("BASE_SIEGE_ROOT", Path.cwd())).expanduser()
ROOT = next((p for p in [candidate, *candidate.parents] if (p / "env/siege_env.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the complete project, or set BASE_SIEGE_ROOT to its folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from env import config as C
from env.siege_env import SiegeEnv, describe_action, deployment_action
from env.render import draw_snapshot
from env.generator import generate_base
from experiments.provenance import runtime_manifest, sha256
RUNS = Path(os.environ.get("BASE_SIEGE_RUNS_DIR", ROOT / "runs")).expanduser().resolve()
OUTPUT = ROOT / "artifacts" / "notebook-sessions" / uuid.uuid4().hex[:12]
OUTPUT.mkdir(parents=True, exist_ok=False)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 12, "axes.labelsize": 10, "font.size": 10})
def save_figure(name):
    plt.savefig(OUTPUT / f"{name}.png", dpi=180, bbox_inches="tight")
    plt.show()
print(f"Project: {ROOT.name} | Python: {sys.version.split()[0]}")
print(f"This notebook's outputs: {OUTPUT}")

## 1. What needs to stay on disk

A Studio JupyterLab space uses an attached EBS volume for its files. Stopping and later starting
the same space preserves that volume. Studio Classic has a different storage layout, so confirm
where your project is stored. Do not use a temporary folder for the only copy of a run.

| Keep together | Purpose |
|---|---|
| Source, notebooks and uv.lock | Recreate the program and packages |
| Pools and registration | Preserve the agreed layouts and protocol |
| Each complete run folder | Recover model, optimiser, games, RNGs and history |
| Evaluation CSVs and manifests | Trace figures back to individual episodes |

Stopping compute is different from deleting a space or its storage. Local checkpoints are not an
independent backup. This project does not copy them to another service.

Source: [AWS Studio JupyterLab spaces](https://docs.aws.amazon.com/sagemaker/latest/dg/studio-updated-jl-user-guide.html).

In [ ]:
import platform
import shutil
import torch
from importlib.metadata import version

packages = ["numpy", "torch", "gymnasium", "stable-baselines3", "sb3-contrib", "nbformat"]
display(pd.DataFrame({"package": packages, "version": [version(p) for p in packages]}))
print("Kernel:", sys.executable)
print("Platform:", platform.platform())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| CUDA:", torch.version.cuda)
print(f"Free project-volume space: {shutil.disk_usage(ROOT).free / 1024**3:.1f} GiB")
assert sys.version_info[:2] == (3, 12), "Use the project's Python 3.12 kernel."

## 2. Choose persistent run storage

The default is the runs folder inside this project. Set BASE_SIEGE_RUNS_DIR before launching
Jupyter only if you need a different persistent location. Avoid temporary folders, container
caches and scratch disks.

This cell tests access; it cannot prove your AWS retention settings. It creates and removes
only its own tiny storage-probe file.

In [ ]:
import tempfile
RUNS.mkdir(parents=True, exist_ok=True)
with tempfile.NamedTemporaryFile(prefix=".storage-probe-", dir=RUNS) as handle:
    handle.write(b"base-siege-storage-check")
    handle.flush()
    os.fsync(handle.fileno())
print("Run directory:", RUNS)
if str(RUNS).startswith(("/tmp/", "/private/tmp/")):
    raise RuntimeError("Choose persistent project storage, not /tmp.")
print("Writable. Confirm volume persistence in Studio.")

## 3. Check the project

Run the complete tests and a short recovery drill in a terminal. They do not start an AWS instance,
create cloud resources, open final layouts or run the full study.

~~~bash
uv run pytest -q
uv run python -m tools.check_recovery --out artifacts/my-recovery-check --kill-process
~~~

Use a new output folder each time. The second command kills only its own short test process,
then compares resumed training against uninterrupted training.

In [ ]:
required = ["env/siege_env.py", "train/checkpoints.py", "train/train_ppo.py",
            "experiments/protocol.py", "uv.lock", "reference/base_siege_stage5_sync.zip"]
inventory = pd.DataFrame({"file": required, "present": [(ROOT / p).is_file() for p in required]})
display(inventory)
assert inventory.present.all()
print("Original ZIP SHA-256:", sha256(ROOT / required[-1]))

## 4. Understand the units

A simulator tick is one movement/combat update. An environment transition is one agent decision.
Shopping consumes decisions but no combat ticks. PPO gathers a rollout from every worker and
then updates the policy using several passes over that data.

The GPU handles neural-network operations. The Python simulator runs on CPUs. This small policy
may be limited by simulation or process overhead, so measure short CPU and CUDA runs on the target
machine before fixing the study setup.

In [ ]:
from train.train_ppo import TrainingSettings, phase_budgets
full = TrainingSettings(placement_steps=C.CURRICULUM_PLACEMENT_TIMESTEPS,
    joint_steps=C.CURRICULUM_JOINT_TIMESTEPS, n_envs=C.N_TRAINING_ENVS,
    rollout_steps=C.PPO_N_STEPS, batch_size=C.PPO_BATCH_SIZE, epochs=C.PPO_N_EPOCHS, mode="full")
display(pd.Series(phase_budgets(full), name="transitions"))
print("Default checkpoint spacing:", phase_budgets(full)["rollout_transitions"], "transitions")

## 5. Stop and resume deliberately

For a planned break, request a pause from a second terminal. Wait for status=paused in the run
manifest, then stop Studio compute. An incomplete rollout is discarded; the last committed
update remains recoverable.

~~~bash
uv run python -m train.train_ppo --request-pause --out runs/P-0
uv run python -m train.train_ppo --resume --out runs/P-0
~~~

After an unexpected shutdown, reopen the same project and resume the same run directory.
The trainer checks saved settings, source, packages and input hashes. A completed run is not
trained again.

With the default cadence, an abrupt stop can lose unsaved work from one rollout and its
in-progress update. The full configuration has 8,192 transitions per rollout. Zero lost work
cannot be promised.

In [ ]:
records = []
for path in sorted(RUNS.glob("*/manifest.json")):
    data = json.loads(path.read_text())
    records.append({"run": path.parent.name, "status": data.get("status"),
        "committed_steps": data.get("actual_steps", 0), "phase": data.get("current_phase"),
        "attempts": len(data.get("attempts", []))})
display(pd.DataFrame(records) if records else pd.DataFrame(columns=["run", "status", "committed_steps", "phase", "attempts"]))

## 6. Before a long run

- Run the recovery lab again in the Studio kernel you will train with.
- Confirm CUDA works if you choose the CUDA device; there is no silent CPU fallback.
- Keep the same source and locked packages throughout the experiment.
- Check disk space. Older checkpoint generations are kept; nothing is pruned automatically.
- Register learner seeds and filtered pools before full training.
- Do not use smoke policies as performance evidence.

Same-machine CPU continuation is tested. Equality across GPU types, drivers or package versions
is not promised. Read [the recovery guide](../docs/SAGEMAKER_STUDIO.md).